# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described via a [Croissant schema](https://mlcommons.org/croissant/) at the following URL:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```


In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'date_published', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")

## 2. Data Overview
Review available _record sets_, _fields_, and their `@id`s using the Croissant metadata. This establishes how the records are structured and referenced later.

In [ ]:
# List available record sets with @id, name, and contained fields

print("Record sets in this dataset:\n")
record_sets = list(metadata.record_sets)
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}")
    print(f"  Name: {getattr(rs, 'name', '-')}\n  Description: {getattr(rs, 'description', '-')}")
    print(f"  Fields (by @id):")
    for field in getattr(rs, 'fields', []):
        print(f"    - {field.id}: {getattr(field, 'name', '')}")
    print()

# Print the raw @id for future reference
available_recordset_ids = [rs.id for rs in record_sets]
print(f"Available RecordSet @ids: {available_recordset_ids}")

Let's view a sample of raw records for each record set using their `@id` keys.

In [ ]:
# Display first record for each record set
for rs in record_sets:
    print(f"Sample record from RecordSet @id={rs.id} ({getattr(rs, 'name', '-')})")
    try:
        for i, item in enumerate(dataset.records(record_set=rs.id)):
            if i >= 1:
                break
            print(item)
    except Exception as e:
        print(f"  [Could not load records: {e}]")
    print("-"*40)


## 3. Data Extraction
Load data from one or more record sets into DataFrames for downstream analysis. _All entities (record sets, fields, columns) are referenced by their `@id` as required_.

In [ ]:
# Extract all available record sets into pandas DataFrames
dataframes = {}

# Use the discovered record set `@id`s
for record_set_id in available_recordset_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for RecordSet {record_set_id}")
        else:
            print(f"No records found in {record_set_id}")
    except Exception as ex:
        print(f"Could not load record set {record_set_id}: {ex}")

if len(dataframes) == 0:
    print("No records set data could be loaded.")
else:
    # Take first loaded record set for demonstration
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nAvailable DataFrame columns for {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())

    print("\nPreview of data:")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing steps:
- Filtering records by a numeric criterion
- Normalizing a numeric field
- Optionally grouping/categorizing records

> **Note**: Please update `numeric_field_id` and `group_field_id` to valid `@id`s from your DataFrame if your dataset structure is complex.

In [ ]:
# For demonstration, auto-select a numeric column if present
import numbers

if len(dataframes):
    df = dataframes[main_record_set_id]
    # Find a likely numeric field
    numeric_field_id = None
    for col in df.columns:
        # Try to cast to float and see if it's mostly numeric
        try:
            if pd.to_numeric(df[col], errors='coerce').notna().sum() > 0.8*len(df):
                numeric_field_id = col
                break
        except Exception:
            continue

    if numeric_field_id is None:
        print("No suitable numeric field found for filtering analysis.")
    else:
        # Ensure column is float
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        print(f"Selected numeric field: {numeric_field_id}")

        # Set threshold for filtering (e.g. mean)
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold:0.2f} (mean): {len(filtered_df)} out of {len(df)}")

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Find a group/categorical column (not numeric, not too many unique vals)
        candidate_group_fields = [col for col in df.columns
                                  if df[col].dtype == 'object' and df[col].nunique() < 15]
        group_field_id = candidate_group_fields[0] if candidate_group_fields else None
        
        if group_field_id is not None:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
            print(grouped_df)
        else:
            print("No suitable group field found for grouping.")
else:
    print("No DataFrames available for EDA.")

## 5. Visualization
Visualize the distribution of a numeric field and, if available, the grouping variable as well.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True, color='navy')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If group_field_id defined, show boxplot
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Not enough data for visualization.")

## 6. Conclusion

- We successfully loaded the dataset metadata and explored available record sets and fields with `mlcroissant`.
- Sample data was extracted via Croissant's `@id` system, allowing dataframes for further analysis.
- Simple exploratory analyses demonstrated filtering by a numeric field, normalizing it, and grouping by a categorical value when available.
- The dataset is ready for advanced analyses, including machine learning workflows, biomarker discovery, or protein group comparisons.